In [44]:
from __future__ import annotations
from typing import Dict,List,Sequence
import random
import math

In [2]:
morse: Dict[str, str] = {'A': '.-', 'B': '-...', 'C': '-.-.', 'D': '-..', 'E': '.'}
chars: List[str] = list(morse.keys())

max_len: int = max(len(s) for s in morse.values())
print(max_len)

4


In [6]:
sym_to_vec : Dict[str, list] = {
    ".":([1.0, 0.0, 0.0]),
    "-":([0.0, 1.0, 0.0]),
    "_":([0.0, 0.0, 1.0]),
}
print(sym_to_vec['.'])
print(sym_to_vec['-'])
print(sym_to_vec["_"])



[1.0, 0.0, 0.0]
[0.0, 1.0, 0.0]
[0.0, 0.0, 1.0]


In [13]:
print(list(sym_to_vec.keys()))
print(list(sym_to_vec.values())) #Cool encoding sequence

['.', '-', '_']
[[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]]


In [22]:
def encode(seq:str) -> list:
    x : List[int] = []
    for i in range(max_len):
        sym: str = seq[i] if i < len(seq) else "_" #'.' → ['.', '_', '_', '_'] or '-...' → ['-', '.', '.', '.'] padding
        x.append(sym_to_vec[sym])
    return sum(x,[])
print(morse["A"])
print(encode(morse["A"])) #Testing  our function
print("dot,dash,padding,padding")


.-
[1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0]
dot,dash,padding,padding


In [42]:
def one_hot(i:int,n:int) -> list:
    y:list = [0.0] * n # create n number of zeroes list.
    y[i] = 1.0 # assign 1.0 to whichever number position i is.
    return y
for __ in range(4): #testing the matrix format
    print(one_hot(__,max_len))

[1.0, 0.0, 0.0, 0.0]
[0.0, 1.0, 0.0, 0.0]
[0.0, 0.0, 1.0, 0.0]
[0.0, 0.0, 0.0, 1.0]


## On to the neural net (Leaky)

In [ ]:
class MLP:
    def __init__(self,n_in:int,n_hidden:int, n_out:int) -> None:
        self.W1: list = [[random.gauss(0,0.4) for ___ in range(n_hidden)] for ___ in range(n_in)] #nested lists for matrix creation
        self.b1 = [0.0] * n_hidden # Hidden-layer biases.
        self.W2 = [[random.gauss(0,0.4) for _ in range(n_out)] for _ in range(n_hidden)] # Hidden-to-output weights.
        self.b2 = [0.0] * n_out # Output-layer biases.

    def leaky_relu(self, z: list, alpha: float = 0.01) -> list: # Leaky is more akin to nature
        return [x if x > 0 else alpha * x for x in z]

    def dleaky_relu(self, z:list, alpha: float = 0.01) -> list:
        return[1 if x > 0 else alpha for x in z] #Leaky ReLU helps prevent "dead neurons" by allowing a small gradient for negative inputs, unlike sigmoid which suffers from vanishing gradients.

    def softmax(self, z: list) -> list: # subtract list of z from its max value, exponentiate and divide by its sum(exp)
        max_val:list = max(z)
        shifted:list = [x - max_val for x in z] #Stabilize
        exps:list = [math.exp(x) for x in shifted] #Smooth
        total:int = sum(exps)
        return [ x/total for x in exps] # Return list of Probabilities

    def forward(self, X: list) -> list: #Forward pass
        # Perform matrix multiplication and add bias
        self.z1:list = [[0 for _ in range(len(self.W1))] for _ in range(len(X))]
        for i in range(len(X)):           # for each row in X
            for j in range(len(self.W1)):      # for each column in W1
                for k in range(len(self.W1)):  # for each row in W1 (or column in X)
                    self.z1[i][j] += X[i][k] * self.W1[k][j]
                self.z1[i][j] += self.b1[j]         # adding the bias after the dot product
        self.a1:list = self.leaky_relu(self.z1)
        self.z2 = [[0 for _ in range(len(self.W2))] for _ in range(len(self.a1))]
        for i in range(len(self.a1)): #forr each leaky relu row a1
            for j in range(len(self.W2)): #for each column in W2
                for k in range(len(self.W2)): #for each row in W2 (
                    self.z2[i][j] += self.a1[i][k] * self.W2[k][j]
                self.z2[j][i] += self.b2[j] #adding the second bias after the dot product
        self.a2:list = self.leaky_relu(self.z2)
        return self.a2

    def loss(self, Yhat:list,Y:list) -> float: #Cross-entropy loss for classification.

        #return float(-np.mean(np.sum(Y * np.log(Yhat + 1e-12), axis=1)))  # Average negative log-likelihood.
        EPS:float = 1e-12
        batch_size:int = len(Y)
        total_loss:float=0.0
        for i in range(batch_size):
            sample_loss=0.0
            for j in range(len(Y[i])):
                sample_loss += Y[i][j] * math.log(Yhat[i][j] + EPS)
            total_loss += -sample_loss
        return total_loss/ batch_size #Return the mean loss

    def train_step(self,X:list,Y:list,lr:float=0.7) -> float:
        n:int = len(X)
        Yhat: list = self.forward(X) # Forward Compute predictions
        dZ2:list = (Yhat-Y)/n # Output-layer error signal for softmax + cross-entropy.
        dW2:list = [[0 for _ in range(len(dZ2))] for _ in range(len(self.a1))]
        for i in range(len(self.a1)):# For each neuron in the hidden layer (row in a1, col in a1.T)
            for j in range(len(dZ2)):# For each neuron in the output layer
                for k in range(len(dZ2)):# For each sample in the batch
                    dW2[i][j] += self.a1[k][i] * dZ2[k][j]# a1.T is a1[k][i] instead of the original (i,k) called transpose
        db2:list = [sum(column) for column in zip(*dZ2)] # Gradient for output biases.
        dA1:list = [[0 for _ in range(len(self.W2))] for _ in range(len(dZ2))]
        for i in range(len(dZ2)): # For each sample in the batch
            for j in range(len(self.W2)): # For each neuron in the hidden layer
                for k in range(len(self.W2)): # For each neuron in the output layer
                    dA1[i][j] += dZ2[i][k]*self.W2[k][j] # W2[k][j] is element (j,k) of W2.T
        dZ1:list=dA1* self.dleaky_relu(self.a1) # Hidden-layer error after relu derivative.
        dW1:list=[[0 for _ in range(len(dZ1))] for _ in range(len(X))]
        for i in range(len(X)): # For each feature in the input
            for j in range(len(dZ1)): # For each neuron in the hidden layer
                for k in range(len(dZ1)): # For each sample in the batch
                    dW1[i][j] += X[k][i] * dZ1[k][j] # X[k][i] is element (i,k) of X.T
        db1:list = [sum(column) for column in zip(*dZ1)] # Gradient for hidden biases.

        self.W2 -= lr * dW2
        self.b2 -= lr * db2
        self.W1 -= lr * dW1
        self.b1 -= lr * db1
        return self.loss(Yhat,Y)





    

